# Creación BD S&P500

La idea es que del dataset con los precios, se tome el último día y se resten 1000 días, después se compare con el otro dataset de tickers para evaluar el sesgo de supervivencia. Los enlaces a dichos datasets se encuentran a continuación:

Dataset con la información de precios: https://www.kaggle.com/datasets/camnugent/sandp500?resource=download

Dataset con la información de tickers: https://github.com/fja05680/sp500

In [78]:
import pandas as pd
import os
import numpy as np
import pandas_market_calendars as mcal


In [79]:
# ---------------------------------------------------------
# BLOQUE 1: Configuración de Rutas y Carga Inicial
# ---------------------------------------------------------
folder_path = r'../data/processed'
path_main = os.path.join(folder_path, 'dataset_with_returns.csv')
path_components = os.path.join(folder_path, 'S&P 500 Historical Components & Changes(01-17-2026).csv')

print("Cargando dataset principal...")
df = pd.read_csv(path_main)
df['date'] = pd.to_datetime(df['date'])

# Aseguramos el orden cronológico para que el recorte de registros sea correcto
df = df.sort_values(['Name', 'date'])

df.head()

Cargando dataset principal...


,date,open,high,low,close,volume,Name,adjusted_price,return_1d
0,2013-02-11,45.17,45.18,44.45,44.60,2915405,A,28.576422,-0.010648
1,2013-02-12,44.81,44.95,44.50,44.62,2373731,A,28.589230,0.000448
2,2013-02-13,44.81,45.24,44.68,44.75,2052338,A,28.672525,0.002914
3,2013-02-14,44.72,44.78,44.36,44.58,3826245,A,28.563612,-0.003799
4,2013-02-15,43.48,44.24,42.21,42.25,14657315,A,27.070713,-0.052266


### Checking if there are dates gaps

In [80]:
def detect_real_gaps(df):
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values(["Name", "date"])

    nyse = mcal.get_calendar("NYSE")

    gaps = []

    for name, group in df.groupby("Name"):
        group = group.sort_values("date").reset_index(drop=True)

        for i in range(1, len(group)):
            prev_date = group.loc[i-1, "date"]
            curr_date = group.loc[i, "date"]

            valid_days = nyse.valid_days(start_date=prev_date, end_date=curr_date)

            # deberían ser 2: prev y next
            if len(valid_days) > 2:
                gaps.append({
                    "Name": name,
                    "prev_date": prev_date,
                    "date": curr_date,
                    "missing_days": valid_days[1:-1]
                })

    return pd.DataFrame(gaps)
    
detect_real_gaps(df).shape

(17, 4)

### Avoiding survivor bias

To eliminate survivor bias, we first obtain all month-end constituent lists 
for the S&P 500 from January 2013 to December 2018. We consolidate these lists 
into a single binary matrix, indicating whether each stock is an index 
constituent in the subsequent month. In this way, we are able to approximately 
reproduce the S&P 500 composition at any point in time within this period. 
In a second step, for all stocks that have been constituents of the index at 
any time during this interval, we download daily total return indices covering 
the same time frame.

In [81]:
folder_path = r'../data/raw'
path_components = os.path.join(folder_path, 'S&P 500 Historical Components & Changes(01-17-2026).csv')
df_hist_comp = pd.read_csv(path_components)
df_hist_comp['date'] = pd.to_datetime(df_hist_comp['date'])

print(df_hist_comp.head())
print("...")
print(df_hist_comp.tail())

        date                                            tickers
0 1996-01-02  AAL,AAMRQ,AAPL,ABI,ABS,ABT,ABX,ACKH,ACV,ADM,AD...
1 1996-01-03  AAL,AAMRQ,AAPL,ABI,ABS,ABT,ABX,ACKH,ACV,ADM,AD...
2 1996-01-04  AAL,AAMRQ,AAPL,ABI,ABS,ABT,ABX,ACKH,ACV,ADM,AD...
3 1996-01-10  AAL,AAMRQ,AAPL,ABI,ABS,ABT,ABX,ACKH,ACV,ADM,AD...
4 1996-01-11  AAL,AAMRQ,AAPL,ABI,ABS,ABT,ABX,ACKH,ACV,ADM,AD...
...
           date                                            tickers
2700 2025-11-11  A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,ADP...
2701 2025-11-28  A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,ADP...
2702 2025-12-11  A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,ADP...
2703 2025-12-22  A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,ADP...
2704 2026-01-14  A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,ADP...


In [82]:
df_hist_comp.dtypes

date       datetime64[us]
tickers               str
dtype: object

In [83]:
df.head()
first_date = df.iloc[0]['date']
print(first_date)

last_date = df.iloc[-1]['date']
print(last_date)

2013-02-11 00:00:00
2018-02-07 00:00:00


Creating pandas tables to see which companies were on S&P 500 at the end of the month

In [84]:
# Filtrar por fechas
df_hist_comp = df_hist_comp[(df_hist_comp['date'] >= first_date) & (df_hist_comp['date'] <= last_date)].copy()

# Extraer año y mes
df_hist_comp['year'] = df_hist_comp['date'].dt.year
df_hist_comp['month'] = df_hist_comp['date'].dt.month

# Tomar último día de cada mes
df_month_end = df_hist_comp.groupby(['year', 'month']).tail(1)

# Eliminar la columna 'date'
df_month_end = df_month_end.drop(columns=['date'])

# Reordenar columnas para que year y month queden primero
cols = ['year', 'month'] + [c for c in df_month_end.columns if c not in ['year', 'month']]
df_month_end = df_month_end[cols]

df_month_end = df_month_end.reset_index(drop=True)

# Revisar resultado
print(df_month_end.head())
print("...")
print(df_month_end.tail())

   year  month                                            tickers
0  2013      2  A,AABA,AAPL,ABBV,ABC,ABT,ACN,ADBE,ADI,ADM,ADP,...
1  2013      3  A,AABA,AAPL,ABBV,ABC,ABT,ACN,ADBE,ADI,ADM,ADP,...
2  2013      4  A,AABA,AAPL,ABBV,ABC,ABT,ACN,ADBE,ADI,ADM,ADP,...
3  2013      5  A,AABA,AAPL,ABBV,ABC,ABT,ACN,ADBE,ADI,ADM,ADP,...
4  2013      6  A,AABA,AAPL,ABBV,ABC,ABT,ACN,ADBE,ADI,ADM,ADP,...
...
    year  month                                            tickers
56  2017     10  A,AAL,AAP,AAPL,ABBV,ABC,ABT,ACN,ADBE,ADI,ADM,A...
57  2017     11  A,AAL,AAP,AAPL,ABBV,ABC,ABT,ACN,ADBE,ADI,ADM,A...
58  2017     12  A,AAL,AAP,AAPL,ABBV,ABC,ABT,ACN,ADBE,ADI,ADM,A...
59  2018      1  A,AAL,AAP,AAPL,ABBV,ABC,ABT,ACN,ADBE,ADI,ADM,A...
60  2018      2  A,AAL,AAP,AAPL,ABBV,ABC,ABT,ACN,ADBE,ADI,ADM,A...


Creating Binary matrix

In [85]:
all_tickers = set()

for tickers_str in df_hist_comp['tickers']:
    for t in tickers_str.split(','):
        all_tickers.add(t.strip())
all_tickers = sorted(all_tickers)

ticker_dict = {ticker: idx for idx, ticker in enumerate(all_tickers)}


n_months = len(df_month_end)
n_tickers = len(all_tickers)

# matriz de ceros
binary_matrix = np.zeros((n_months, n_tickers), dtype=int)

for i, row in df_month_end.iterrows():
    tickers_in_row = row['tickers'].split(',')  # separa los tickers
    for t in tickers_in_row:
        t = t.strip()  # quitar espacios
        if t in ticker_dict:  # seguridad
            j = ticker_dict[t]  # columna correspondiente
            binary_matrix[i, j] = 1


binary_df = pd.DataFrame(binary_matrix, columns=all_tickers)
binary_df.insert(0, 'year', df_month_end['year'])
binary_df.insert(1, 'month', df_month_end['month'])

print(binary_df.head())

   year  month  A  AABA  AAL  AAP  AAPL  ABBV  ABC  ABT  ...  XL  XLNX  XOM  \
0  2013      2  1     1    0    0     1     1    1    1  ...   1     1    1   
1  2013      3  1     1    0    0     1     1    1    1  ...   1     1    1   
2  2013      4  1     1    0    0     1     1    1    1  ...   1     1    1   
3  2013      5  1     1    0    0     1     1    1    1  ...   1     1    1   
4  2013      6  1     1    0    0     1     1    1    1  ...   1     1    1   

   XRAY  XRX  XYL  YUM  ZBH  ZION  ZTS  
0     1    1    1    1    1     1    0  
1     1    1    1    1    1     1    0  
2     1    1    1    1    1     1    0  
3     1    1    1    1    1     1    0  
4     1    1    1    1    1     1    1  

[5 rows x 619 columns]


### Create rolling windows

#### Paso 1: Definir los bloques de 1000 dias

In [86]:
df_rolling_windows = df  # tu DataFrame de precios diarios


all_dates = pd.bdate_range(start=df['date'].min(), end=df['date'].max())

training_days = 750
trading_days = 250
study_period_days = training_days + trading_days

start_idx = 0
study_periods = []

while start_idx + study_period_days <= len(all_dates):
    period_dates = all_dates[start_idx:start_idx + study_period_days]
    
    study_periods.append({
        '1000_days_block_dates': period_dates,
    })
    
    start_idx += trading_days  # rolling cada 250 días

print(len(study_periods))


2


Solo tendremos dos periodos de estudio

#### Paso 2: Filtrar tickers usando la matriz binaria

This function is made so it search the companies that were on S&P500 the last month of the 1000 blocks day

In [87]:
def get_tickers_for_period(binary_df, last_train_date):
    year = last_train_date.year
    month = last_train_date.month
    
    row = binary_df[
        (binary_df['year'] == year) &
        (binary_df['month'] == month)
    ]
    
    if row.empty:
        return []
    
    tickers_series = row.drop(columns=['year', 'month']).iloc[0]
    tickers = tickers_series[tickers_series == 1].index.tolist()
    
    return tickers


Create study periods with metadata

In [88]:
study_periods_data = []

for i, period in enumerate(study_periods):
    
    last_train_date = pd.to_datetime(period['1000_days_block_dates'][-250])
    
    tickers = get_tickers_for_period(binary_df, last_train_date)
    
    # 🔹 Filtrar training
    df_1000_days = df_rolling_windows[
        (df_rolling_windows['Name'].isin(tickers)) &
        (df_rolling_windows['date'].isin(period['1000_days_block_dates']))
    ].copy()
    

    # Guardar todo
    study_periods_data.append({
        'period_id': i,
        'tickers': tickers,
        'n_i': len(tickers),
        '1000_days_df': df_1000_days,
        '1000_days_start': period['1000_days_block_dates'][0],
        '1000_days_end': period['1000_days_block_dates'][-1],
    })



In [89]:
for sp in study_periods_data:
    print(f"\nStudy Period {sp['period_id']}")
    print(f"n_i: {sp['n_i']}")
    print(f"data range: {sp['1000_days_start']} → {sp['1000_days_end']}")
    print(f"#data rows: {len(sp['1000_days_df'])}")



Study Period 0
n_i: 502
data range: 2013-02-11 00:00:00 → 2016-12-09 00:00:00
#data rows: 419582

Study Period 1
n_i: 506
data range: 2014-01-27 00:00:00 → 2017-11-24 00:00:00
#data rows: 448161


### Create rolling windows for each stock

In [90]:

import numpy as np

WINDOW_SIZE = 240

def calculate_returns(df, price_col="price"):
    
    df["return_1d"] = df['return_1d']

    return df

def create_sequences_for_stock_s(df):
    df = calculate_returns(df)
    sequences_for_ticker = []

    window_size = WINDOW_SIZE

    for i in range(window_size, len(df)):
        
        window = df.iloc[i - window_size+1:i+1]["return_1d"].values
        
        window = window.reshape(-1, 1)
        sequences_for_ticker.append(window)

    return sequences_for_ticker
         

def build_sequences_dataset_for_study_period(sp):
    sequences = []
    tickers = sp.get('tickers', [])
    df_1000_days = sp.get('1000_days_df')
    
    for t in tickers:
        df_ticker = df_1000_days[df_1000_days["Name"] == t]
        sequences_for_ticker = create_sequences_for_stock_s(df_ticker)
        sequences.append(sequences_for_ticker)

    return sequences


for sp in study_periods_data:
    sequences_dataset_built = build_sequences_dataset_for_study_period(sp)
    sp["sequences_dataset"] = sequences_dataset_built
    

In [91]:
def debug_sequences(sp, max_tickers=3, max_sequences=2):
    sequences = sp.get("sequences_dataset", [])
    tickers = sp.get("tickers", [])

    print(f"\nStudy Period {sp['period_id']}")
    print(f"Total tickers: {len(sequences)}")
    for i, (ticker, ticker_sequences) in enumerate(zip(tickers, sequences)):
        if i >= max_tickers:
            break

        print(f"\nTicker: {ticker}")
        print(f"Total sequences: {len(ticker_sequences)}")

        for j, seq in enumerate(ticker_sequences[:max_sequences]):
            print(f"  Seq {j} shape: {seq.shape}")
            print(f"  First 3 values: {seq[:3].flatten()}")

debug_sequences(study_periods_data[0])


Study Period 0
Total tickers: 502

Ticker: A
Total sequences: 727
  Seq 0 shape: (240, 1)
  First 3 values: [ 0.0004482   0.00291354 -0.00379853]
  Seq 1 shape: (240, 1)
  First 3 values: [ 0.00291354 -0.00379853 -0.05226576]

Ticker: AABA
Total sequences: 0

Ticker: AAL
Total sequences: 727
  Seq 0 shape: (240, 1)
  First 3 values: [-0.01313991  0.02733006 -0.04570259]
  Seq 1 shape: (240, 1)
  First 3 values: [ 0.02733006 -0.04570259  0.03645488]


We note that there are som Tickers/Stock that do not have sequences. Probably because they were not on the original data

In [92]:
def get_empty_sequence_tickers(sp):
    sequences = sp.get("sequences_dataset", [])
    tickers = sp.get("tickers", [])

    empty_tickers = []

    for ticker, ticker_sequences in zip(tickers, sequences):
        if len(ticker_sequences) == 0:
            empty_tickers.append(ticker)

    return empty_tickers


In [93]:
def inspect_missing_tickers(empty_tickers, df):
    for t in empty_tickers:
        df_t = df[df["Name"] == t]

        print(f"\nTicker: {t}")
        print(f"Rows in original df: {len(df_t)}")

        if len(df_t) > 0:
            print(f"Date range: {df_t['date'].min()} → {df_t['date'].max()}")
            print(f"Unique dates: {df_t['date'].nunique()}")
        else:
            
            print("⚠️ Not present in original dataset")

for sp in study_periods_data:
    print(f"\nStudy Period {sp['period_id']}")
    empty = get_empty_sequence_tickers(sp)
    inspect_missing_tickers(empty, df)


Study Period 0

Ticker: AABA
Rows in original df: 0
⚠️ Not present in original dataset

Ticker: ADT
Rows in original df: 0
⚠️ Not present in original dataset

Ticker: AN
Rows in original df: 0
⚠️ Not present in original dataset

Ticker: APTV
Rows in original df: 43
Date range: 2017-12-06 00:00:00 → 2018-02-07 00:00:00
Unique dates: 43

Ticker: ARG
Rows in original df: 0
⚠️ Not present in original dataset

Ticker: BBBY
Rows in original df: 0
⚠️ Not present in original dataset

Ticker: BCR
Rows in original df: 0
⚠️ Not present in original dataset

Ticker: BHGE
Rows in original df: 151
Date range: 2017-07-05 00:00:00 → 2018-02-07 00:00:00
Unique dates: 151

Ticker: BKNG
Rows in original df: 0
⚠️ Not present in original dataset

Ticker: BRCM
Rows in original df: 0
⚠️ Not present in original dataset

Ticker: BXLT
Rows in original df: 0
⚠️ Not present in original dataset

Ticker: CAM
Rows in original df: 0
⚠️ Not present in original dataset

Ticker: CBRE
Rows in original df: 0
⚠️ Not presen

We will remove those not present in the original dataset.

In [94]:
def get_invalid_tickers(empty_tickers, df, min_len=WINDOW_SIZE):
    invalid = []

    for t in empty_tickers:
        df_t = df[df["Name"] == t]

        # no está en el dataset o no tiene suficientes datos
        if len(df_t) == 0 or len(df_t) < min_len:
            invalid.append(t)

    return invalid

def remove_invalid_tickers_from_sp(sp, invalid_tickers):
    tickers = sp.get("tickers", [])
    sequences = sp.get("sequences_dataset", [])

    new_tickers = []
    new_sequences = []

    for t, seq in zip(tickers, sequences):
        if t not in invalid_tickers:
            new_tickers.append(t)
            new_sequences.append(seq)

    sp["tickers"] = new_tickers
    sp["sequences_dataset"] = new_sequences

    return 

for sp in study_periods_data:
    empty = get_empty_sequence_tickers(sp)
    invalid = get_invalid_tickers(empty, df)
    remove_invalid_tickers_from_sp(sp, invalid)

### Build lstm dataset

In [95]:
def sequences_to_dataframe(sp):
    sequences = sp.get("sequences_dataset", [])
    tickers = sp.get("tickers", [])

    rows = []

    # nombres de columnas: R_-239 ... R_0
    start_window = WINDOW_SIZE -1
    col_names = [f"R_{-i}" for i in reversed(range(WINDOW_SIZE))]
    
    for ticker, ticker_sequences in zip(tickers, sequences):
        for seq in ticker_sequences:
            seq_flat = seq.flatten() 

            row = dict(zip(col_names, seq_flat))
            row["ticker"] = ticker

            rows.append(row)

    df_sequences = pd.DataFrame(rows)

    return df_sequences



In [96]:
for sp in study_periods_data:
    df_sequences = sequences_to_dataframe(sp)
    sp["df_sequences"] = df_sequences


In [97]:
for sp in study_periods_data:
    print(f"\nStudy Period {sp['period_id']}")
    df_sequences = sp.get("df_sequences")
    print(df_sequences.shape)


Study Period 0
(313982, 241)

Study Period 1
(335500, 241)


### Creating targets

Following the methodology proposed by Takeuchi and Lee (2013), we formulate a **binary classification problem** based on stock returns.

For each stock \( s \) and each time step \( t \), we define a target variable \( Y_{t+1}^s \) that depends on the return in the next period.

The procedure is as follows:

1. For each day \( t+1 \), we collect the one-period returns \( R_{t+1}^{1,s} \) of **all stocks**.
2. These returns are sorted in ascending order.
3. The **cross-sectional median** of the returns is computed.
4. Each stock is assigned to a class:
   - **Class 0**: if the stock return is **below** the median.
   - **Class 1**: if the stock return is **greater than or equal to** the median.

This results in two balanced classes at each time step, allowing the problem to be framed as a binary classification task where the goal is to predict whether a stock will perform **below or above the market average** in the next period.

In [98]:
def add_target_with_pandas(df_sequences):
    df = df_sequences.copy()

    # 1. future return
    df["R_t+1"] = df.groupby("ticker")["R_0"].shift(-1)

    # remove NaNs
    df = df[df["R_t+1"].notna()].copy()

    # 2. time index
    df["time_idx"] = df.groupby("ticker").cumcount()

    # 3. compute quantiles per time
    q_low = df.groupby("time_idx")["R_t+1"].transform(lambda x: x.quantile(0.3))
    q_high = df.groupby("time_idx")["R_t+1"].transform(lambda x: x.quantile(0.7))

    # 4. assign classes
    df["target"] = 0  # HOLD

    df.loc[df["R_t+1"] <= q_low, "target"] = -1  # SELL
    df.loc[df["R_t+1"] >= q_high, "target"] = 1  # BUY

    # cleanup
    df = df.drop(columns=["R_t+1", "time_idx"])

    return df


In [99]:
for sp in study_periods_data:
    df_sequences = sp.get("df_sequences")
    df_sequences_with_target = add_target_with_pandas(df_sequences)
    sp["df_sequences"] = df_sequences_with_target

In [ ]:

print(f"\nStudy Period {sp['period_id']}")
df_sequences = study_periods_data[0].get("df_sequences")
df_sequences.head()

In [ ]:
df_sequences.to_csv("../data/processed/lstm_240_sequences_dataset.csv", index=False)

### Sequences dataset for Shallow Neural Network

For this neural network, we will use only 31 days back because this kind of network doenst have the ability of the lSTM networks. it cant remember large patrons

In [ ]:
def build_dnn_features_from_lags(df):
    """
    Convert lag-based dataset (R_-239 ... R_0) into 31 features
    following Fischer & Krauss.

    IMPORTANT:
    - The original columns (R_-k) represent individual daily returns at time t-k.
    - The new features R_k DO NOT represent single-day returns.
    - Instead, each R_k is an ACCUMULATED return over the last k days.

    Meaning:
    - R_1  = return at time t (latest return)
    - R_5  = sum of returns from t-4 to t
    - R_20 = sum of returns from t-19 to t
    - R_240 = sum of returns from t-239 to t

    So we are transforming:
        raw time series (lags)
    into:
        aggregated features (cumulative returns over different horizons)

    This is required for DNN (memory-free model), unlike LSTM,
    which uses the full sequence without aggregation.
    """

    df = df.copy()

    # windows from paper
    windows = list(range(1, 21)) + list(range(40, 241, 20))

    # get lag columns ordered correctly (oldest → newest)
    lag_cols = [col for col in df.columns if col.startswith("R_-") or col == "R_0"]

    # sort columns numerically (R_-239 ... R_0)
    lag_cols = sorted(lag_cols, key=lambda x: int(x.replace("R_", "")))

    # build accumulated features
    for w in windows:
        # take last w columns (most recent values)
        selected_cols = lag_cols[-w:]

        # cumulative return over last w days
        df[f"R_{w}"] = df[selected_cols].sum(axis=1)

    # select final columns
    feature_cols = [f"R_{w}" for w in windows]
    final_cols = feature_cols + ["ticker", "target"]

    return df[final_cols]


df_snn = build_dnn_features_from_lags(df_sequences)

In [ ]:
df_snn.head()

In [ ]:
df_snn.shape

In [ ]:
df_snn.to_csv("../data/processed/lstm_snn_sequences_dataset.csv", index=False)